# FLIMKit vs Leica LAS X — Fitting Comparison
Runs FLIMKit summed-decay fits on all PTU files in `forAlex/` and compares the results side-by-side with the Leica `_fit.xlsx` exports.

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')

# Notebook plot readability: force dark text on light backgrounds
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'text.color': 'black',
    'axes.labelcolor': 'black',
    'axes.titlecolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black',
})

# -- make FLIMKit importable ------------------------------------------------
FLIMKIT_ROOT = Path('/Users/as-hunt/FLIMKit')
if str(FLIMKIT_ROOT) not in sys.path:
    sys.path.insert(0, str(FLIMKIT_ROOT))

from flimkit.PTU.reader import PTUFile
from flimkit.FLIM.irf_tools import irf_from_xlsx_analytical, gaussian_irf_from_fwhm
from flimkit.FLIM.fitters import fit_summed
from flimkit.utils.xlsx_tools import load_xlsx
from flimkit.configs import (n_exp, Tau_min, Tau_max, de_population, de_maxiter, lm_restarts, MACHINE_IRF_FIT_BG,
                              MACHINE_IRF_FIT_SIGMA, MACHINE_IRF_FIT_TAIL)
from flimkit.interactive import _load_machine_irf_prompt

MACHINE_IRF_DEFAULT_PATH = Path('/Users/as-hunt/FLIMKit/flimkit/machine_irf/machine_irf_CRUKS.npy')
FOLDER = Path('/Users/as-hunt/Downloads/forAlex/')
N_EXP  = 3   # <- change if you used a different model in Leica
print('Ready.')

Ready.


In [2]:
# ── helpers ───────────────────────────────────────────────────────────────

def parse_leica_fit(xlsx_path: Path) -> dict:
    """Return {parameter_string: value} from a Leica *_fit.xlsx."""
    df = pd.read_excel(xlsx_path)
    return {str(r['Parameter']).strip(): r['Value']
            for _, r in df.iterrows() if pd.notna(r.get('Parameter'))}


STRATEGIES = {
    'Machine': dict(label='Machine IRF (CRUKS)', color='#4CAF50', marker='s'),
}


def _build_irf(strategy: str, ptu, decay, xlsx_path):
    """Return (irf_prompt, has_tail, fit_bg, fit_sigma) for the given strategy."""
    if strategy == 'Machine':
        peak_bin = int(np.argmax(decay))
        irf, _ = _load_machine_irf_prompt(None, ptu.n_bins, peak_bin)
        return irf, MACHINE_IRF_FIT_TAIL, MACHINE_IRF_FIT_BG, MACHINE_IRF_FIT_SIGMA

    raise ValueError(f'Unknown strategy: {strategy}')


def run_flimkit_fit(ptu_path: Path, xlsx_path: Path | None,
                    nexp: int, strategy: str = 'Machine') -> dict | None:
    """Run FLIMKit summed fit on channel 1 (FLIM FRET) with the given IRF strategy."""
    try:
        ptu   = PTUFile(str(ptu_path), verbose=False)
        decay = ptu.summed_decay(channel=1)  # Explicit channel 1: FLIM FRET
        irf, has_tail, fit_bg, fit_sigma = _build_irf(strategy, ptu, decay, xlsx_path)
        if irf is None:
            return None
        _, summary = fit_summed(
            decay, ptu.tcspc_res, ptu.n_bins,
            irf, has_tail, fit_bg, fit_sigma,
            nexp, Tau_min, Tau_max,
            optimizer='de', n_restarts=lm_restarts,
            de_popsize=de_population, de_maxiter=de_maxiter,
            workers=-1, polish=True, cost_function='poisson',
        )
        return summary
    except Exception as e:
        print(f'  ERROR [{strategy}] {ptu_path.name}: {e}')
        return None

print('Helpers defined. Fitting on channel 1 (FLIM FRET) with Machine IRF (CRUKS).')


Helpers defined. Fitting on channel 1 (FLIM FRET) with Machine IRF (CRUKS).


In [3]:
# ── run all fits ──────────────────────────────────────────────────────────
ptu_files = sorted(FOLDER.glob('*.ptu'))
print(f'Found {len(ptu_files)} PTU files,  strategies: {list(STRATEGIES)}\n')

rows = []

for ptu_path in ptu_files:
    stem      = ptu_path.stem
    fit_xlsx  = FOLDER / f'{stem}_fit.xlsx'
    comp_xlsx = FOLDER / f'{stem}.xlsx'
    xlsx_path = comp_xlsx if comp_xlsx.exists() else None

    print(f'── {stem}')
    row = {'File': stem}

    # ── FLIMKit: all three strategies ────────────────────────────────────
    for strat, meta in STRATEGIES.items():
        fk = run_flimkit_fit(ptu_path, xlsx_path, N_EXP, strategy=strat)
        if fk is None:
            print(f'   [{strat}]  skipped / failed')
            continue
        pfx = strat
        row[f'{pfx} τ_amp (ns)'] = float(fk['tau_mean_amp_ns'])
        row[f'{pfx} τ_int (ns)'] = float(fk['tau_mean_int_ns'])
        row[f'{pfx} χ²r']        = float(fk['reduced_chi2_tail_pearson'])
        for i in range(N_EXP):
            row[f'{pfx} τ{i+1} (ns)'] = float(fk['taus_ns'][i])
        print(f'   [{strat:10s}]  τ_int={row[f"{pfx} τ_int (ns)"]:.3f} ns  '
              f'τ_amp={row[f"{pfx} τ_amp (ns)"]:.3f} ns  χ²r={row[f"{pfx} χ²r"]:.3f}')

    # ── Leica reference ──────────────────────────────────────────────────
    lc = parse_leica_fit(fit_xlsx) if fit_xlsx.exists() else {}
    row['LC τ_amp (ns)'] = float(lc.get('Mean τ, Amplitude Weighted [ns]', np.nan))
    row['LC τ_int (ns)'] = float(lc.get('Mean τ, Intensity Weighted  [ns]', np.nan))
    row['LC χ²']         = float(lc.get('χ²', np.nan))
    for i in range(N_EXP):
        row[f'LC τ{i+1} (ns)'] = float(lc.get(f'Lifetime (τ) {i+1} [ns]', np.nan))
    print(f'   [Leica     ]  τ_int={row["LC τ_int (ns)"]:.3f} ns  '
          f'τ_amp={row["LC τ_amp (ns)"]:.3f} ns  χ²={row["LC χ²"]:.3f}\n')

    rows.append(row)

df = pd.DataFrame(rows)
print('Done!')
df.round(4)

Found 10 PTU files,  strategies: ['Machine']

── R 1 (4) z1 s1 t1
  Cost function: poisson
  bg initial guess = 235.000 cts/bin, upper bound = 470.000 (free param)
  σ broadening: fixed at 0
  Fit window: bins 1–266 (0.10–25.79 ns), 265 bins
  Differential evolution: popsize=30, maxiter=5000, workers=-1
  Running final LM polish...
   [Machine   ]  τ_int=1.830 ns  τ_amp=1.169 ns  χ²r=5.397
   [Leica     ]  τ_int=1.798 ns  τ_amp=1.037 ns  χ²=1.343

── R 1 (4) z1 s2 t115
  Cost function: poisson
  bg initial guess = 84.000 cts/bin, upper bound = 168.000 (free param)
  σ broadening: fixed at 0
  Fit window: bins 1–275 (0.10–26.67 ns), 274 bins
  Differential evolution: popsize=30, maxiter=5000, workers=-1
  Running final LM polish...
   [Machine   ]  τ_int=1.841 ns  τ_amp=1.161 ns  χ²r=6.356
   [Leica     ]  τ_int=1.797 ns  τ_amp=1.053 ns  χ²=1.633

── R 1 (4) z1 s2 t230
  Cost function: poisson
  bg initial guess = 107.000 cts/bin, upper bound = 214.000 (free param)
  σ broadening: fixed

,File,Machine τ_amp (ns),Machine τ_int (ns),Machine χ²r,Machine τ1 (ns),Machine τ2 (ns),Machine τ3 (ns),LC τ_amp (ns),LC τ_int (ns),LC χ²,LC τ1 (ns),LC τ2 (ns),LC τ3 (ns)
0,R 1 (4) z1 s1 t1,1.1691,1.8302,5.3969,3.0334,1.5099,0.4690,1.037,1.798,1.343,0.263,1.025,2.671
1,R 1 (4) z1 s2 t115,1.1609,1.8406,6.3556,3.3869,1.6515,0.4632,1.053,1.797,1.633,0.301,1.147,2.750
2,R 1 (4) z1 s2 t230,1.1886,1.8664,3.7102,3.0373,1.4793,0.4533,1.096,1.841,1.476,0.318,1.172,2.790
3,R 1 (4) z2 s1 t115,1.1568,1.8300,6.3401,3.1735,1.5827,0.4542,1.061,1.800,1.524,0.307,1.171,2.769
4,R 1 (4) z2 s1 t175,1.1872,1.8628,9.1400,3.4049,1.6998,0.4799,1.093,1.823,1.872,0.334,1.226,2.810
5,R 1 (4) z2 s2 t200,1.2035,1.9003,6.4518,3.3405,1.6619,0.4706,1.109,1.867,1.525,0.326,1.225,2.878
6,R 1 (4) z2 s2 t75,1.1545,1.8271,4.9611,3.1531,1.5708,0.4605,1.076,1.799,1.985,0.334,1.191,2.792
7,R 1 (4) z3 s1 t115,1.1773,1.8765,7.4734,3.3920,1.6706,0.4647,1.065,1.832,1.668,0.286,1.093,2.756
8,R 1 (4) z3 s1 t230,1.3624,2.0924,5.1352,3.4181,1.7218,0.4915,1.260,2.052,1.519,0.345,1.292,2.991
9,R 1 (4) z3 s2 t1,1.1547,1.8830,4.2553,3.4630,1.6052,0.4439,1.066,1.839,1.769,0.318,1.209,2.890


In [4]:
# ── scatter: τ_int and τ_amp — Machine IRF vs Leica ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (metric, title) in zip(
        axes,
        [('τ_int', 'Intensity-weighted mean τ'),
         ('τ_amp', 'Amplitude-weighted mean τ')]):

    lc_col = f'LC {metric} (ns)'
    strat = 'Machine'
    col = f'{strat} {metric} (ns)'
    
    x = df[lc_col].dropna()
    y = df[col].dropna()
    
    valid = x.tolist() + y.tolist()
    lims = [min(valid) * 0.93, max(valid) * 1.07]
    
    ax.scatter(x, y, s=80, alpha=0.6, color='#4CAF50', edgecolor='black', linewidth=1.5)
    ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.5, label='y=x (perfect)')
    ax.set_xlabel(f'Leica {metric} (ns)', fontsize=11)
    ax.set_ylabel(f'FLIMKit {metric} (ns)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.grid(True, alpha=0.35)
    ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


In [5]:
# ── scatter: individual components τ1, τ2, τ3 ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

strat = 'Machine'
for i, ax in enumerate(axes):
    fk_col = f'{strat} τ{i+1} (ns)'
    lc_col = f'LC τ{i+1} (ns)'
    
    if fk_col not in df.columns:
        ax.text(0.5, 0.5, f'τ{i+1}: no data', ha='center', va='center')
        continue
    
    x = df[lc_col].dropna()
    y = df[fk_col].dropna()
    
    valid = x.tolist() + y.tolist()
    lims = [min(valid) * 0.85, max(valid) * 1.15]
    
    ax.scatter(x, y, s=80, alpha=0.6, color='#4CAF50', edgecolor='black', linewidth=1.5)
    ax.plot(lims, lims, 'k--', lw=1, alpha=0.5)
    ax.set_xlabel(f'Leica τ{i+1} (ns)', fontsize=11)
    ax.set_ylabel(f'FLIMKit τ{i+1} (ns)', fontsize=11)
    ax.set_title(f'Component {i+1}', fontsize=11, fontweight='bold')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()


In [6]:
# ── bar: Δτ = FLIMKit − Leica for mean lifetimes ──────────────────────
strat = 'Machine'
delta_rows = []
for _, row in df.iterrows():
    r = {'File': row['File'].replace('R 1 (4) ', '')}
    for metric in ['τ_int', 'τ_amp']:
        fk_col = f'{strat} {metric} (ns)'
        lc_col = f'LC {metric} (ns)'
        r[f'Δ{metric}'] = row[fk_col] - row[lc_col]
    delta_rows.append(r)

delta_df = pd.DataFrame(delta_rows).set_index('File')

fig, ax = plt.subplots(figsize=(14, 5))
x_pos = np.arange(len(delta_df))
width = 0.35

bars1 = ax.bar(x_pos - width/2, delta_df['Δτ_int'], width, label='Δτ_int', color='#1976D2', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x_pos + width/2, delta_df['Δτ_amp'], width, label='Δτ_amp', color='#F57C00', alpha=0.8, edgecolor='black')

ax.axhline(0, color='black', lw=1.5, linestyle='-', alpha=0.7)
ax.set_xlabel('File', fontsize=11, fontweight='bold')
ax.set_ylabel('Δτ (FLIMKit − Leica) [ns]', fontsize=11, fontweight='bold')
ax.set_title('Lifetime difference vs Leica (target: close to 0)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(delta_df.index, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f'Δτ_int: mean={delta_df["Δτ_int"].mean():.4f} ns, std={delta_df["Δτ_int"].std():.4f} ns')
print(f'Δτ_amp: mean={delta_df["Δτ_amp"].mean():.4f} ns, std={delta_df["Δτ_amp"].std():.4f} ns')


Δτ_int: mean=0.0361 ns, std=0.0072 ns
Δτ_amp: mean=0.0999 ns, std=0.0148 ns


In [7]:
# ── grouped bar: component τ per scan, all strategies + Leica ────────────
short       = df['File'].str.replace('R 1 (4) ', '', regex=False).tolist()
all_sources = list(STRATEGIES.items()) + [('LC', {'label': 'Leica', 'color': '#9E9E9E', 'marker': 'D'})]
n_sources   = len(all_sources)
total_w     = 0.85
bar_w       = total_w / n_sources

fig, axes = plt.subplots(1, N_EXP, figsize=(6 * N_EXP, 5), sharey=False)
if N_EXP == 1:
    axes = [axes]

x       = np.arange(len(df))
offsets = np.linspace(-total_w/2 + bar_w/2, total_w/2 - bar_w/2, n_sources)

for i, ax in enumerate(axes):
    for (src, meta), off in zip(all_sources, offsets):
        col = f'{src} τ{i+1} (ns)'
        if col not in df.columns:
            continue
        ax.bar(x + off, df[col], bar_w,
               label=meta['label'], color=meta['color'], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(short, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('τ (ns)', fontsize=10)
    ax.set_title(f'τ{i+1} per scan', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Component Lifetimes per Scan — all strategies vs Leica', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FOLDER / 'comparison_tau_per_scan.png', dpi=150, bbox_inches='tight')
plt.show()


In [8]:
# ── summary table: Machine IRF χ²r and Leica comparison ──────────────────
strat = 'Machine'
summary_rows = []
for _, row in df.iterrows():
    r = {'File': row['File'].replace('R 1 (4) ', '')}
    r['τ_int (ns)'] = round(row.get(f'{strat} τ_int (ns)', np.nan), 4)
    r['τ_amp (ns)'] = round(row.get(f'{strat} τ_amp (ns)', np.nan), 4)
    r['χ²r (tail, Pearson)'] = round(row.get(f'{strat} χ²r', np.nan), 3)
    r['Leica τ_int (ns)'] = round(row.get('LC τ_int (ns)', np.nan), 4)
    r['Leica χ²'] = round(row.get('LC χ²', np.nan), 3)
    r['Δτ_int (ns)'] = round(row.get(f'{strat} τ_int (ns)', np.nan) - row.get('LC τ_int (ns)', np.nan), 4)
    summary_rows.append(r)

summary_df = pd.DataFrame(summary_rows).set_index('File')

summary_df.style \
    .background_gradient(subset=['Δτ_int (ns)'], cmap='RdYlGn_r', vmin=-0.5, vmax=0.5) \
    .format('{:.4f}', na_rep='—') \
    .set_caption('Machine IRF (CRUKS) on channel 1 (FLIM FRET)  |  Δ = FLIMKit − Leica  |  green = close match')


,τ_int (ns),τ_amp (ns),"χ²r (tail, Pearson)",Leica τ_int (ns),Leica χ²,Δτ_int (ns)
File,,,,,,
z1 s1 t1,1.8302,1.1691,5.3970,1.7980,1.3430,0.0322
z1 s2 t115,1.8406,1.1609,6.3560,1.7970,1.6330,0.0436
z1 s2 t230,1.8664,1.1886,3.7100,1.8410,1.4760,0.0254
z2 s1 t115,1.8300,1.1568,6.3400,1.8000,1.5240,0.0300
z2 s1 t175,1.8628,1.1872,9.1400,1.8230,1.8720,0.0398
z2 s2 t200,1.9003,1.2035,6.4520,1.8670,1.5250,0.0333
z2 s2 t75,1.8271,1.1545,4.9610,1.7990,1.9850,0.0281
z3 s1 t115,1.8765,1.1773,7.4730,1.8320,1.6680,0.0445
z3 s1 t230,2.0924,1.3624,5.1350,2.0520,1.5190,0.0404


In [9]:
# ── save CSV ──────────────────────────────────────────────────────────────
out = FOLDER / 'flimkit_vs_leica_comparison.csv'
df.round(5).to_csv(out, index=False)
print(f'Saved → {out}')

Saved → /Users/as-hunt/Downloads/forAlex/flimkit_vs_leica_comparison.csv
